# 01 — Controlled RGB-only 12-frame model

This is **not** the old exploratory 12-frame experiment. It is a new controlled temporal-resolution ablation built from the same canonical 29-frame strokes and the same grouped source-video split as the 29-frame models.

- 12 frames = 4 buildup / 3 execution / 5 follow-through.
- I3D features were cached in Notebook 00.
- The model predicts the full 3×5 target matrix.
- Phase weights are used only when reporting an overall score, **not in the training loss**.
- Three fixed seeds are run for reproducibility.

In [ ]:
from pathlib import Path
import os, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from scipy.stats import spearmanr, pearsonr

REVISION_ROOT = Path(os.environ.get("AQA_REVISION_ROOT", str(Path.cwd() / "artifacts"))).expanduser().resolve()
CACHE_DIR = REVISION_ROOT / "cache"
RESULTS_DIR = REVISION_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(CACHE_DIR / "manifest_with_split.csv")
y_all = np.load(CACHE_DIR / "y_scores.npy")
train_idx = np.load(CACHE_DIR / "train_idx.npy")
val_idx = np.load(CACHE_DIR / "val_idx.npy")
test_idx = np.load(CACHE_DIR / "test_idx.npy")

assert len(manifest) == len(y_all)
print("Samples:", len(manifest))
print("Split:", len(train_idx), len(val_idx), len(test_idx))

PHASE_WEIGHTS = np.asarray([0.25, 0.50, 0.25], dtype=np.float32)
PHASE_NAMES = ["Buildup", "Execution", "FollowThrough"]
BODY_PARTS = ["Head", "Shoulders", "Hands", "Hips", "Feet"]

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def safe_spearman(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(spearmanr(a, b).correlation)

def safe_pearson(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(pearsonr(a, b)[0])

def overall_scalar(y_phase, weights=PHASE_WEIGHTS):
    phase_scalar = np.asarray(y_phase).mean(axis=2)
    overall = np.sum(phase_scalar * np.asarray(weights)[None, :], axis=1)
    return phase_scalar, overall

def calculate_metrics(y_true, y_pred):
    true_phase, true_overall = overall_scalar(y_true)
    pred_phase, pred_overall = overall_scalar(y_pred)

    result = {
        "overall_SRC": safe_spearman(true_overall, pred_overall),
        "overall_Pearson": safe_pearson(true_overall, pred_overall),
        "overall_MAE": float(np.mean(np.abs(true_overall - pred_overall))),
        "overall_RMSE": float(np.sqrt(np.mean((true_overall - pred_overall)**2))),
    }

    for p, name in enumerate(PHASE_NAMES):
        result[f"{name}_SRC"] = safe_spearman(true_phase[:, p], pred_phase[:, p])
        result[f"{name}_Pearson"] = safe_pearson(true_phase[:, p], pred_phase[:, p])
        result[f"{name}_MAE"] = float(np.mean(np.abs(true_phase[:, p] - pred_phase[:, p])))

    return result

def common_head(z, name_prefix):
    # Common latent size/head across all ablation models.
    z = layers.Dense(
        128, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_project"
    )(z)
    z = layers.Dropout(0.25, name=f"{name_prefix}_project_dropout")(z)
    z = layers.Dense(
        96, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_dense96"
    )(z)
    z = layers.Dropout(0.35, name=f"{name_prefix}_drop96")(z)
    z = layers.Dense(
        48, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name=f"{name_prefix}_dense48"
    )(z)
    z = layers.Dropout(0.25, name=f"{name_prefix}_drop48")(z)
    return layers.Dense(5, activation="linear", name=f"{name_prefix}_scores")(z)

def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="mse",
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

def make_callbacks(run_dir):
    return [
        callbacks.EarlyStopping(
            monitor="val_loss", patience=12, restore_best_weights=True, min_delta=1e-3
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=str(run_dir / "best.weights.h5"),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ]

def save_run(model_name, seed, model, history, y_train, pred_train, y_val, pred_val, y_test, pred_test):
    run_dir = RESULTS_DIR / model_name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    metrics = {"model": model_name, "seed": int(seed)}
    for split_name, yt, yp in [
        ("train", y_train, pred_train),
        ("validation", y_val, pred_val),
        ("test", y_test, pred_test),
    ]:
        mm = calculate_metrics(yt, yp)
        metrics.update({f"{split_name}_{k}": v for k, v in mm.items()})

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    metrics["best_epoch"] = best_epoch

    pd.DataFrame(history.history).to_csv(run_dir / "history.csv", index=False)
    pd.DataFrame([metrics]).to_csv(run_dir / "metrics.csv", index=False)

    np.savez_compressed(
        run_dir / "predictions.npz",
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
        y_train=y_train, pred_train=pred_train,
        y_val=y_val, pred_val=pred_val,
        y_test=y_test, pred_test=pred_test,
    )

    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "model": model_name,
                "seed": int(seed),
                "optimizer": "Adam",
                "learning_rate": 3e-4,
                "loss": "MSE",
                "epochs_max": 120,
                "batch_size": 32,
                "early_stopping_patience": 12,
                "reduce_lr_patience": 5,
                "phase_weights_used_for_reporting_only": [0.25, 0.50, 0.25],
                "note": "Model predicts the 3x5 phase/body-part matrix directly. Phase weights are not used in training loss."
            },
            f,
            indent=2,
        )
    return metrics

In [ ]:
rgb12 = np.load(CACHE_DIR / "rgb12_i3d_norm.npy", mmap_mode="r")
print("RGB-12 feature tensor:", rgb12.shape)

x_train = np.asarray(rgb12[train_idx], dtype=np.float32)
x_val = np.asarray(rgb12[val_idx], dtype=np.float32)
x_test = np.asarray(rgb12[test_idx], dtype=np.float32)

PHASE_SLICES_12 = [(0, 4), (4, 7), (7, 12)]

def build_rgb12_model():
    feat_dim = x_train.shape[-1]
    inp = layers.Input(shape=(12, feat_dim), name="rgb12_i3d")
    x = layers.LayerNormalization(name="rgb12_ln")(inp)

    outs = []
    for p, (s, e) in enumerate(PHASE_SLICES_12):
        pooled = layers.GlobalAveragePooling1D(name=f"phase{p}_rgb_pool")(x[:, s:e, :])
        outs.append(common_head(pooled, f"phase{p}"))

    return models.Model(inp, layers.Lambda(lambda z: tf.stack(z, axis=1), name="phase_scores")(outs))

model_check = build_rgb12_model()
print("Output shape:", model_check.output_shape)
assert model_check.output_shape == (None, 3, 5)

In [ ]:
SEEDS = [42, 123, 2026]
all_run_metrics = []

y_train = y_all[train_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

for seed in SEEDS:
    print("\n" + "="*80)
    print("MODEL:", "RGB12_GROUPED", "| SEED:", seed)
    print("="*80)

    tf.keras.backend.clear_session()
    set_seed(seed)

    run_dir = RESULTS_DIR / "RGB12_GROUPED" / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    model = build_rgb12_model()
    model = compile_model(model)
    model.summary()

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=120,
        batch_size=32,
        callbacks=make_callbacks(run_dir),
        verbose=1,
        shuffle=True,
    )

    pred_train = model.predict(x_train, batch_size=64, verbose=0)
    pred_val = model.predict(x_val, batch_size=64, verbose=0)
    pred_test = model.predict(x_test, batch_size=64, verbose=0)

    metrics = save_run(
        "RGB12_GROUPED", seed, model, history,
        y_train, pred_train, y_val, pred_val, y_test, pred_test
    )
    all_run_metrics.append(metrics)

summary_df = pd.DataFrame(all_run_metrics)
display(summary_df)

summary_path = RESULTS_DIR / "RGB12_GROUPED" / "all_seeds_metrics.csv"
summary_df.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)